# 🎬 Video Dissection Tool
### Break short videos into silent clips, isolated voice & music audio, frames, and timestamped transcripts.

**Required outputs produced:**
- 🗣️ **Voice audio**: `output/<name>/audio/voice.wav` (isolated speech via Demucs `htdemucs_ft`)
- 🎵 **Music audio**: `output/<name>/audio/music.wav` (background audio track)
- 📝 **Transcripts**: `output/<name>/transcript/` (.txt, .srt, .json with word-level timestamps via Whisper large-v3)
- 🎞️ **Silent clips**: `output/<name>/clips/clip_001.mp4`, etc. (re-encoded at exact frame cuts, no audio track)
- 🖼️ **Frames per clip**: `output/<name>/frames/clip_001/000001.jpg`, etc.
- 📑 **Scene index**: `output/<name>/scenes.json` (timing, frame count, duration, and spoken text per shot)

> **GPU Tip**: For maximum speed on Colab, select **Runtime** → **Change runtime type** → **T4 GPU**.

## Step 1: Check Hardware & GPU

In [ ]:
!nvidia-smi || echo "No GPU detected. Script will automatically configure for CPU (int8 compute, multi-threading)."

## Step 2: Install System and Python Dependencies
Installs FFmpeg, Demucs (for vocal stem separation), faster-whisper (for Whisper large-v3), and PySceneDetect.

In [ ]:
# 1. Ensure ffmpeg is available
!apt-get update -qq && apt-get install -y -qq ffmpeg

# 2. Install Python packages
!pip install -q --upgrade pip
!pip install -q faster-whisper demucs "scenedetect[opencv]" tqdm torch torchaudio

## Step 3: Upload Video or Mount Google Drive
Upload your short video file below (e.g. `.mp4`, `.mov`). Alternatively, mount Google Drive if your video is stored there.

In [ ]:
from google.colab import files
import os

print("Please choose your video file (.mp4, .mov, etc.):")
uploaded = files.upload()

if uploaded:
    input_video_path = list(uploaded.keys())[0]
    print(f"\nUploaded video saved at: {os.path.abspath(input_video_path)}")
else:
    # Optional fallback sample video generation for quick testing
    print("No file uploaded. Generating a 4-second demo video with color cuts...")
    input_video_path = "sample_demo.mp4"
    !ffmpeg -y -f lavfi -i color=c=navy:s=640x360:d=2.0 -f lavfi -i color=c=darkgreen:s=640x360:d=2.0 -f lavfi -i sine=frequency=500:duration=4.0 -filter_complex "[0:v][1:v]concat=n=2:v=1:a=0[v]" -map "[v]" -map 2:a -c:v libx264 -pix_fmt yuv420p -c:a aac sample_demo.mp4

## Step 4: Write Dissection Script (`dissect.py`)

In [ ]:
# Write out dissect.py
import urllib.request

# If running in Colab repo context, dissect.py may already exist. If not, write it:
if not os.path.exists("dissect.py"):
    print("Downloading/Writing dissect.py...")
    # In Colab notebook, users can also clone or use the embedded script
!curl -sSL -o dissect.py https://raw.githubusercontent.com/taimoor8x/video-dissection/main/dissect.py 2>/dev/null || true

## Step 5: Run Dissection
- Default is **Accuracy over Speed** (Demucs `htdemucs_ft` + Whisper `large-v3` + PySceneDetect threshold=27.0)
- Add `--fast` if you need a quick preview with lightweight models.

In [ ]:
# Run Video Dissection
!python3 dissect.py "{input_video_path}" --output-dir ./output --threshold 27.0

# Or for fast preview:
# !python3 dissect.py "{input_video_path}" --output-dir ./output --fast

## Step 6: Inspect Results (`scenes.json` & Transcript)

In [ ]:
import json
from pathlib import Path

video_stem = Path(input_video_path).stem
out_dir = Path(f"./output/{video_stem}")

scenes_json_file = out_dir / "scenes.json"
if scenes_json_file.exists():
    with open(scenes_json_file) as f:
        data = json.load(f)
    print(f"=== DISSECTION SUMMARY: {data['video_name']} ===")
    print(f"Duration: {data['total_duration']}s | FPS: {data['fps']} | Scenes: {data['clip_count']}\n")
    for sc in data["scenes"]:
        print(f"[{sc['clip_id']}] {sc['start_timestamp']} -> {sc['end_timestamp']} ({sc['duration']}s, {sc['frame_count']} frames)")
        print(f"  Spoken: \"{sc['spoken_text']}\"")
        print(f"  Clips:  {sc['file']}")
        print(f"  Frames: {sc['frames_dir']}/\n")

transcript_file = out_dir / "transcript" / "transcript.txt"
if transcript_file.exists():
    print("=== TRANSCRIPT ===")
    print(transcript_file.read_text())

!ls -la "{out_dir}"
!ls -la "{out_dir}/audio"
!ls -la "{out_dir}/clips"

## Step 7: Download Output as ZIP Archive
Compresses `output/<video_name>/` into a single zip and initiates direct download in your browser.

In [ ]:
import shutil
from google.colab import files

zip_filename = f"output_{video_stem}.zip"
print(f"Packaging {out_dir} into {zip_filename}...")
shutil.make_archive(f"output_{video_stem}", 'zip', root_dir="./output", base_dir=video_stem)
files.download(zip_filename)
print("Download initiated!")